In [3]:
import numpy as np, pandas as pd, torch, gc
from torch_geometric.data import Data
from sklearn.neighbors import BallTree
from sklearn.preprocessing import LabelEncoder, StandardScaler

CSV_PATH  = "crime_data_engineered.csv"
SAVE_PATH = "crime_graph_knn10.pt"
K         = 10              # 10 nearest neighbours  → 2·N·K ≈ 5.4 M edges
EARTH_R   = 6_371_000.0     # metres (haversine scale)


In [4]:
df = pd.read_csv(CSV_PATH)

le = LabelEncoder()
df["Primary_Type_enc"] = le.fit_transform(df["Primary Type"])

feat_cols = [
    "Latitude", "Longitude",
    "Hour_sin", "Hour_cos",
    "Month_sin", "Month_cos",
    "Crime_MA_7D", "Crime_MA_30D",
    "Primary_Type_enc",
]
features = df[feat_cols].copy()

scaler = StandardScaler()
features[["Crime_MA_7D", "Crime_MA_30D"]] = scaler.fit_transform(
    features[["Crime_MA_7D", "Crime_MA_30D"]]
)

x = torch.tensor(features.values, dtype=torch.float)        # [N, 9]


In [5]:
# --- spatial neighbours (undirected) -------------------------------
coords_rad = np.deg2rad(df[["Latitude", "Longitude"]].values)
tree       = BallTree(coords_rad, metric="haversine")

dist_rad, ind = tree.query(coords_rad, k=K + 1)        # self + k
dist_m   = (dist_rad[:, 1:] * EARTH_R).ravel()          # drop self
nbr_idx  = ind[:, 1:].ravel()
src_idx  = np.repeat(np.arange(len(df)), K)

edge_src = np.concatenate([src_idx, nbr_idx]).astype(np.int32)
edge_dst = np.concatenate([nbr_idx, src_idx]).astype(np.int32)
edge_dist= np.concatenate([dist_m,  dist_m ]).astype(np.float32)

# --- timestamps as plain NumPy int32 ---------------------------------
t0 = pd.to_datetime(df["Date"]) + pd.to_timedelta(df["Hour"], unit="h")

# convert the whole array and its min to int64 first, THEN divide
ts_int64 = t0.values.astype("int64")            # nanoseconds since epoch
epoch    = ts_int64.min()                       # scalar int64
ts       = ((ts_int64 - epoch) // 10**9).astype(np.int32)   # seconds, int32


# --- Δt vectorised (fits easily in RAM) ------------------------------
edge_time = np.abs(ts[edge_src] - ts[edge_dst]).astype(np.float32)

# --- torch tensors ---------------------------------------------------
edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
edge_attr  = torch.tensor(
    np.vstack([edge_dist, edge_time]).T, dtype=torch.float32
)

print(f"Edges (k={K}): {edge_index.size(1):,}   "
      f"edge_attr ≈ {edge_attr.element_size()*edge_attr.nelement()/1e6:.1f} MB")


Edges (k=10): 5,445,280   edge_attr ≈ 43.6 MB


C:\Users\atuly\AppData\Local\Temp\ipykernel_12364\3281826555.py:27: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:257.)
  edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)


In [6]:
y_hotspot = torch.tensor(df[["Latitude", "Longitude"]].values, dtype=torch.float)
y_type    = torch.tensor(df["Primary_Type_enc"].values,          dtype=torch.long)
y_hour    = torch.tensor(df["Hour"].values,                      dtype=torch.long)

data = Data(
    x          = x,
    edge_index = edge_index,
    edge_attr  = edge_attr,
    y_hotspot  = y_hotspot,
    y_type     = y_type,
    y_hour     = y_hour,
)

torch.save(data, SAVE_PATH)
print("✅ graph saved as", SAVE_PATH)
print(data)


✅ graph saved as crime_graph_knn10.pt
Data(x=[272264, 9], edge_index=[2, 5445280], edge_attr=[5445280, 2], y_hotspot=[272264, 2], y_type=[272264], y_hour=[272264])


In [7]:
import torch, numpy as np

import torch

data = torch.load("crime_graph_knn10.pt",
                  weights_only=False,    # ← allow full un-pickle
                  map_location="cpu")    # (optional) keep on CPU




rng  = np.random.default_rng(seed=42)
perm = rng.permutation(data.num_nodes)

n_train = int(0.80 * data.num_nodes)
n_val   = int(0.10 * data.num_nodes)

train_idx = perm[:n_train]
val_idx   = perm[n_train:n_train+n_val]
test_idx  = perm[n_train+n_val:]

data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.val_mask   = torch.zeros_like(data.train_mask)
data.test_mask  = torch.zeros_like(data.train_mask)

data.train_mask[train_idx] = True
data.val_mask[val_idx]     = True
data.test_mask[test_idx]   = True

print("✔  masks added:",
      data.train_mask.sum().item(),
      data.val_mask.sum().item(),
      data.test_mask.sum().item())


✔  masks added: 217811 27226 27227


In [23]:
import torch, numpy as np
from torch_geometric.loader import NeighborLoader

# 1 ── load the saved graph ----------------------------------------------------
data = torch.load("crime_graph_knn10.pt", weights_only=False)

# 2 ── compute index splits (80/10/10) ----------------------------------------
rng   = np.random.default_rng(seed=42)
perm  = torch.as_tensor(rng.permutation(data.num_nodes), dtype=torch.long)

n_train = int(0.80 * data.num_nodes)
n_val   = int(0.10 * data.num_nodes)

train_idx = perm[:n_train]                 # 217 k indices
val_idx   = perm[n_train:n_train+n_val]    # 27 k
test_idx  = perm[n_train+n_val:]           # 27 k

print("Index splits:",
      train_idx.numel(), val_idx.numel(), test_idx.numel())

# 3 ── build NeighborLoaders using *indices* ----------------------------------
fan_out     = [10, 7, 5]
batch_size  = 4096

train_loader = NeighborLoader(
    data,
    num_neighbors = fan_out,
    batch_size    = batch_size,
    input_nodes   = train_idx,   # ← index tensor, not mask
    shuffle       = True,
    num_workers   = 0,           # 0 is safest on Windows/CPU
)

val_loader = NeighborLoader(
    data,
    num_neighbors = fan_out,
    batch_size    = batch_size,
    input_nodes   = val_idx,
    shuffle       = False,
    num_workers   = 0,
)

print("✔ loaders ready; preview batch:")
print(next(iter(train_loader)))


Index splits: 217811 27226 27227
✔ loaders ready; preview batch:
Data(x=[100607, 9], edge_index=[2, 451908], edge_attr=[451908, 2], y_hotspot=[100607, 2], y_type=[100607], y_hour=[100607], n_id=[100607], e_id=[451908], num_sampled_nodes=[4], num_sampled_edges=[3], input_id=[4096], batch_size=4096)


In [32]:
# ── Load graph  →  make index splits  →  build loaders (no masks) ────────────
import torch, numpy as np
from torch_geometric.loader import NeighborLoader

# 1) load the saved graph
data = torch.load("crime_graph_knn10.pt", weights_only=False)

# 2) create index splits (train / val / test)
rng  = np.random.default_rng(seed=42)
perm = torch.as_tensor(rng.permutation(data.num_nodes), dtype=torch.long)

n_train = int(0.80 * data.num_nodes)
n_val   = int(0.10 * data.num_nodes)

train_idx = perm[:n_train]                # 217 k indices
val_idx   = perm[n_train:n_train+n_val]   #  27 k
test_idx  = perm[n_train+n_val:]          #  27 k

print("Index splits:", train_idx.numel(), val_idx.numel(), test_idx.numel())

# 3) build NeighborLoaders using those index tensors
fan_out    = [5, 5, 5]     # 3-hop fan-out
batch_size = 4096           # seed nodes per step

train_loader = NeighborLoader(
    data,
    num_neighbors = fan_out,
    batch_size    = 1024,
    input_nodes   = train_idx,   # ← index tensor, not a mask
    shuffle       = True,
    num_workers   = 0,           # 0 is safest on Windows/CPU
)

val_loader = NeighborLoader(
    data,
    num_neighbors = fan_out,
    batch_size    = 1024,
    input_nodes   = val_idx,
    shuffle       = False,
    num_workers   = 0,
)

print("✔ loaders ready — preview first train batch:")
print(next(iter(train_loader)))


Index splits: 217811 27226 27227
✔ loaders ready — preview first train batch:
Data(x=[24495, 9], edge_index=[2, 75135], edge_attr=[75135, 2], y_hotspot=[24495, 2], y_type=[24495], y_hour=[24495], n_id=[24495], e_id=[75135], num_sampled_nodes=[4], num_sampled_edges=[3], input_id=[1024], batch_size=1024)


In [39]:
# --- dataset-level check -------------------------------------------------
print("NaNs in node features :", torch.isnan(data.x).sum().item())
print("NaNs in y_hotspot     :", torch.isnan(data.y_hotspot).sum().item())
print("NaNs in y_type        :", torch.isnan(data.y_type).sum().item())
print("NaNs in y_hour        :", torch.isnan(data.y_hour).sum().item())


NaNs in node features : 2
NaNs in y_hotspot     : 0
NaNs in y_type        : 0
NaNs in y_hour        : 0


In [40]:
# Replace NaN / ±inf in x with 0
data.x = torch.nan_to_num(data.x, nan=0.0, posinf=0.0, neginf=0.0)

# verify
print("NaNs after fix :", torch.isnan(data.x).sum().item())   # should print 0


NaNs after fix : 0


In [48]:
import torch
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class SAGESpatialTemporal(nn.Module):
    def __init__(self, in_channels, hidden, num_types):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden, aggr='mean')
        self.conv2 = SAGEConv(hidden,      hidden, aggr='mean')
        self.lin_hot  = nn.Linear(hidden, 2)
        self.lin_type = nn.Linear(hidden, num_types)
        self.lin_hour = nn.Linear(hidden, 2)   # sin–cos
    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        return self.lin_hot(h), self.lin_type(h), self.lin_hour(h)


In [49]:
def haversine_rmse(pred, true):
    R = 6_371_000.0
    φ1, φ2 = torch.deg2rad(true[:, 0]), torch.deg2rad(pred[:, 0])
    dφ     = φ2 - φ1
    dλ     = torch.deg2rad(pred[:, 1] - true[:, 1])
    a = torch.sin(dφ/2)**2 + torch.cos(φ1)*torch.cos(φ2)*torch.sin(dλ/2)**2
    return (R*2*torch.arcsin(a.clamp(max=1).sqrt())).pow(2).mean().sqrt()


In [51]:
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
num_types = int(data.y_type.max()) + 1
model     = SAGESpatialTemporal(data.num_node_features, hidden=32,
                                num_types=num_types).to(device)
opt       = torch.optim.Adam(model.parameters(), lr=5e-4)

def haversine_rmse(pred, true):
    R = 6_371_000.0
    φ1, φ2 = torch.deg2rad(true[:,0]), torch.deg2rad(pred[:,0])
    dφ, dλ = φ2-φ1, torch.deg2rad(pred[:,1]-true[:,1])
    a = torch.sin(dφ/2)**2 + torch.cos(φ1)*torch.cos(φ2)*torch.sin(dλ/2)**2
    return (R*2*torch.arcsin(a.clamp(max=1).sqrt())).pow(2).mean().sqrt()

for epoch in range(1, 21):
    # ---- train --------------------------------------------------------
    model.train();  tot = 0
    for batch in train_loader:
        batch = batch.to(device)
        hot, typ, hr = model(batch.x, batch.edge_index)

        seeds  = torch.arange(batch.batch_size, device=device)
        h_true = batch.y_hour[seeds].float()
        target = torch.stack([torch.sin(2*torch.pi*h_true/24),
                              torch.cos(2*torch.pi*h_true/24)], 1)

        Lhot  = F.mse_loss(hot[seeds],  batch.y_hotspot[seeds])
        Ltype = F.cross_entropy(typ[seeds], batch.y_type[seeds])
        Lhour = F.mse_loss(hr[seeds],  target)

        loss = 0.5*Lhot + 1.0*Ltype + 1.0*Lhour
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()

    # ---- validation ---------------------------------------------------
    model.eval();  v_loss = 0; v_rmse = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            hot, typ, hr = model(batch.x, batch.edge_index)

            seeds  = torch.arange(batch.batch_size, device=device)
            h_true = batch.y_hour[seeds].float()
            target = torch.stack([torch.sin(2*torch.pi*h_true/24),
                                  torch.cos(2*torch.pi*h_true/24)], 1)

            v_loss += (
                0.5*F.mse_loss(hot[seeds],  batch.y_hotspot[seeds]) +
                F.cross_entropy(typ[seeds], batch.y_type[seeds])   +
                F.mse_loss(hr[seeds], target)
            ).item()
            v_rmse += haversine_rmse(hot[seeds].cpu(),
                                     batch.y_hotspot[seeds].cpu()).item()
    print(f"Epoch {epoch:02d} | "
          f"train {tot/len(train_loader):.4f} | "
          f"val {v_loss/len(val_loader):.4f} | "
          f"RMSE {v_rmse/len(val_loader)/1000:.2f} km")


Epoch 01 | train 590.9918 | val 3.6377 | RMSE 129.56 km
Epoch 02 | train 3.5107 | val 3.3802 | RMSE 105.43 km
Epoch 03 | train 3.2674 | val 3.1579 | RMSE 82.57 km
Epoch 04 | train 3.0637 | val 2.9718 | RMSE 60.74 km
Epoch 05 | train 2.8943 | val 2.8218 | RMSE 43.30 km
Epoch 06 | train 2.7622 | val 2.6996 | RMSE 32.99 km
Epoch 07 | train 2.6528 | val 2.5981 | RMSE 28.92 km
Epoch 08 | train 2.5552 | val 2.5013 | RMSE 28.21 km
Epoch 09 | train 2.4604 | val 2.4086 | RMSE 28.36 km
Epoch 10 | train 2.3683 | val 2.3133 | RMSE 28.53 km
Epoch 11 | train 2.2717 | val 2.2179 | RMSE 28.53 km
Epoch 12 | train 2.1743 | val 2.1261 | RMSE 28.21 km
Epoch 13 | train 2.0771 | val 2.0252 | RMSE 27.31 km
Epoch 14 | train 1.9796 | val 1.9340 | RMSE 26.41 km
Epoch 15 | train 1.8851 | val 1.8377 | RMSE 25.97 km
Epoch 16 | train 1.7923 | val 1.7436 | RMSE 25.88 km
Epoch 17 | train 1.7041 | val 1.6596 | RMSE 26.36 km
Epoch 18 | train 1.6158 | val 1.5727 | RMSE 27.35 km
Epoch 19 | train 1.5331 | val 1.4905 | RMS

In [54]:
import torch
from torch_geometric.loader import NeighborLoader

# ------------------------------------------------------------------
# helper that returns per-node distance (km), *not* averaged
def haversine_km_vec(pred, true):
    R = 6_371_000.0
    φ1, φ2 = torch.deg2rad(true[:, 0]), torch.deg2rad(pred[:, 0])
    dφ     = φ2 - φ1
    dλ     = torch.deg2rad(pred[:, 1] - true[:, 1])
    a = torch.sin(dφ/2)**2 + torch.cos(φ1)*torch.cos(φ2)*torch.sin(dλ/2)**2
    return R * 2 * torch.arcsin(torch.clamp(a.sqrt(), max=1.0)) / 1000  # km
# ------------------------------------------------------------------

test_loader = NeighborLoader(
    data, num_neighbors=fan_out, batch_size=1024,
    input_nodes=test_idx, shuffle=False, num_workers=0
)

model.eval()
sum_sq   = 0.0
n_total  = 0
type_ok  = 0
hour_ok  = 0

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        hot, typ, hr = model(batch.x, batch.edge_index)

        seeds = torch.arange(batch.batch_size, device=device)

        # 1️⃣ hotspot RMSE
        dist = haversine_km_vec(hot[seeds].cpu(),
                                batch.y_hotspot[seeds].cpu())  # [batch]
        sum_sq  += (dist ** 2).sum().item()
        n_total += seeds.numel()

        # 2️⃣ crime type acc
        type_ok += (typ[seeds].argmax(1) == batch.y_type[seeds]).sum().item()

        # 3️⃣ hour acc (sin–cos → class)
        angle     = torch.atan2(hr[seeds][:, 0], hr[seeds][:, 1])
        pred_hour = ((angle / (2 * torch.pi)) * 24) % 24
        hour_ok  += (pred_hour.round().long() == batch.y_hour[seeds]).sum().item()

rmse_km  = (sum_sq / n_total) ** 0.5
type_acc = type_ok / n_total
hour_acc = hour_ok / n_total

print(f"\nTest hotspot RMSE  {rmse_km:.2f} km")
print(f"Test type accuracy {type_acc:.3%}")
print(f"Test hour accuracy {hour_acc:.3%}")



Test hotspot RMSE  29.81 km
Test type accuracy 61.101%
Test hour accuracy 49.032%


In [55]:
torch.save(model.state_dict(), "sage_spatiotemporal_best.pt")


In [73]:
class SAGESpatialTemporal(nn.Module):
    def __init__(self, in_channels, hidden, num_types):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden)
        self.conv2 = SAGEConv(hidden,      hidden)
        self.conv3 = SAGEConv(hidden,      hidden)
        self.lin_hot  = nn.Linear(hidden, 2)
        self.lin_type = nn.Linear(hidden, num_types)
        self.lin_hour = nn.Linear(hidden, 2)   # predict [sin, cos]

    def forward(self, x, edge_index):     # <- NO edge_weight now
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        h = F.relu(self.conv3(h, edge_index))
        return self.lin_hot(h), self.lin_type(h), self.lin_hour(h)


In [75]:
# -- create weighted CE for crime type balancing --
hist = torch.bincount(data.y_type)
w_type = (1.0 / hist.float()).to(device)
w_type = w_type / w_type.mean()

print("✓ Created w_type for", len(w_type), "crime classes.")


✓ Created w_type for 36 crime classes.


In [76]:
model = SAGESpatialTemporal(data.num_node_features, hidden=96,
                             num_types=int(data.y_type.max())+1).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

best_val = 1e9
patience = 5
bad = 0
best_state = None

for epoch in range(1, 61):
    model.train(); tot = 0
    for batch in train_loader:
        batch = batch.to(device)

        hot, typ, hr = model(batch.x, batch.edge_index)   # NO edge_weight here
        seeds = torch.arange(batch.batch_size, device=device)

        h_true = batch.y_hour[seeds].float()
        target_hr = torch.stack([torch.sin(2*math.pi*h_true/24),
                                 torch.cos(2*math.pi*h_true/24)], 1)

        loss = (0.5 * F.mse_loss(hot[seeds], batch.y_hotspot[seeds]) +
                F.cross_entropy(typ[seeds], batch.y_type[seeds], weight=w_type) +
                1.0 * F.mse_loss(hr[seeds], target_hr))
        
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        tot += loss.item()

    # validation
    model.eval(); val = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            hot, typ, hr = model(batch.x, batch.edge_index)   # NO edge_weight here

            seeds = torch.arange(batch.batch_size, device=device)
            h_true = batch.y_hour[seeds].float()
            target_hr = torch.stack([torch.sin(2*math.pi*h_true/24),
                                     torch.cos(2*math.pi*h_true/24)], 1)

            val += (0.5 * F.mse_loss(hot[seeds], batch.y_hotspot[seeds]) +
                    F.cross_entropy(typ[seeds], batch.y_type[seeds], weight=w_type) +
                    1.0 * F.mse_loss(hr[seeds], target_hr)).item()

    val /= len(val_loader)
    print(f"Epoch {epoch:02d}  val {val:.4f}")

    if val < best_val - 1e-3:
        best_val, best_state, bad = val, model.state_dict(), 0
    else:
        bad += 1
        if bad == patience:
            break

model.load_state_dict(best_state)
torch.save(model.state_dict(), "sage3_latlon_best.pt")
print("✔️ Early-stopped at epoch", epoch)


Epoch 01  val 3.9496
Epoch 02  val 2.5717
Epoch 03  val 2.5526
Epoch 04  val 1.7215
Epoch 05  val 1.6720
Epoch 06  val 1.8234
Epoch 07  val 1.5861
Epoch 08  val 1.3264
Epoch 09  val 1.7605
Epoch 10  val 1.2384
Epoch 11  val 1.0846
Epoch 12  val 1.3123
Epoch 13  val 0.9361
Epoch 14  val 0.9597
Epoch 15  val 0.8385
Epoch 16  val 0.8990
Epoch 17  val 0.9760
Epoch 18  val 1.9255
Epoch 19  val 1.0152
Epoch 20  val 0.8770
✔️ Early-stopped at epoch 20


In [77]:
model.eval()
rmse_sum = 0.0
type_correct = 0
hour_correct = 0
n_total = 0

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        hot, typ, hr = model(batch.x, batch.edge_index)

        seeds = torch.arange(batch.batch_size, device=device)

        dist = haversine_km_vec(hot[seeds].cpu(), batch.y_hotspot[seeds].cpu())
        rmse_sum += (dist ** 2).sum().item()
        n_total += seeds.numel()

        type_correct += (typ[seeds].argmax(1) == batch.y_type[seeds]).sum().item()

        angle = torch.atan2(hr[seeds][:,0], hr[seeds][:,1])
        pred_hour = ((angle / (2*math.pi)) * 24) % 24
        hour_correct += (pred_hour.round().long() == batch.y_hour[seeds]).sum().item()

rmse = (rmse_sum / n_total) ** 0.5
type_acc = type_correct / n_total
hour_acc = hour_correct / n_total

print(f"\nTest hotspot RMSE: {rmse:.2f} km")
print(f"Test crime-type accuracy: {type_acc:.3%}")
print(f"Test peak-hour accuracy: {hour_acc:.3%}")



Test hotspot RMSE: 63.60 km
Test crime-type accuracy: 87.197%
Test peak-hour accuracy: 65.284%


In [90]:
import torch

data.y_grid = torch.tensor(df["grid_class"].values, dtype=torch.long)
print("✅ y_grid attached:", data.y_grid.shape)


✅ y_grid attached: torch.Size([272264])


In [91]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch

# Load full CSV with same length as your graph
df = pd.read_csv("crime_data_engineered.csv", usecols=["Latitude", "Longitude"])

# Align with your graph's node count
df = df.iloc[:data.num_nodes].reset_index(drop=True)

# Define grid resolution (≈1 km at mid-latitude)
GRID_RES = 0.01  # degrees

# Discretize lat/lon
df["lat_bin"] = ((df["Latitude"]  - df["Latitude"].min())  / GRID_RES).astype(int)
df["lon_bin"] = ((df["Longitude"] - df["Longitude"].min()) / GRID_RES).astype(int)

# Combine into unique grid keys
df["grid_id"] = df["lat_bin"].astype(str) + "_" + df["lon_bin"].astype(str)

# Encode as class labels
grid_le = LabelEncoder()
df["grid_class"] = grid_le.fit_transform(df["grid_id"])

# Attach to graph
data.y_grid = torch.tensor(df["grid_class"].values, dtype=torch.long)
n_cells = len(grid_le.classes_)

print(f"✅ Attached y_grid of shape: {data.y_grid.shape}")
print(f"✅ Total grid cell classes: {n_cells}")


✅ Attached y_grid of shape: torch.Size([272264])
✅ Total grid cell classes: 695


In [94]:
from torch_geometric.loader import NeighborLoader

# These were likely already created earlier, just repeating for clarity
fan_out = [5, 5, 5]
batch_size = 512

# Assuming you already have these index tensors:
# train_idx, val_idx, test_idx

train_loader = NeighborLoader(
    data,
    input_nodes=train_idx,
    num_neighbors=fan_out,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = NeighborLoader(
    data,
    input_nodes=val_idx,
    num_neighbors=fan_out,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

test_loader = NeighborLoader(
    data,
    input_nodes=test_idx,
    num_neighbors=fan_out,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print("✅ Loaders rebuilt successfully.")


✅ Loaders rebuilt successfully.


In [97]:
batch = next(iter(train_loader))
print("✅ batch.y_grid shape:", batch.y_grid.shape)


✅ batch.y_grid shape: torch.Size([12517])


In [95]:
class SAGEGridClassifier(nn.Module):
    def __init__(self, in_channels, hidden, num_types, n_cells):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.conv3 = SAGEConv(hidden, hidden)
        self.lin_grid = nn.Linear(hidden, n_cells)         # grid cell class
        self.lin_type = nn.Linear(hidden, num_types)       # crime type
        self.lin_hour = nn.Linear(hidden, 2)               # hour sin/cos

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        h = F.relu(self.conv3(h, edge_index))
        return self.lin_grid(h), self.lin_type(h), self.lin_hour(h)


In [96]:
# Forward pass
grid_logits, typ_logits, hr = model(batch.x, batch.edge_index)
seeds = torch.arange(batch.batch_size, device=device)

# Hour → sin/cos
h_true = batch.y_hour[seeds].float()
target_hr = torch.stack([torch.sin(2 * math.pi * h_true / 24),
                         torch.cos(2 * math.pi * h_true / 24)], 1)

# Compute losses
L_grid = F.cross_entropy(grid_logits[seeds], batch.y_grid[seeds])
L_type = F.cross_entropy(typ_logits[seeds], batch.y_type[seeds], weight=w_type)
L_hour = F.mse_loss(hr[seeds], target_hr)

# Total loss
loss = L_grid + L_type + L_hour


AttributeError: 'GlobalStorage' object has no attribute 'y_grid'